# Лабораторная работа 5 — автоматизация SSH и мониторинг SNMP

В этой работе два Linux-based сетевых узла запущены в Containerlab. Нужно программно настроить лабораторный канал, проверить связность и получить системные данные через SNMP.

Итоговая схема: `r1:eth1 (10.50.0.1/30) ↔ (10.50.0.2/30) eth1:s1`.

## 1. Подготовка

Код ниже одинаково работает в Codespaces и в namespace, созданном Clabgate.

In [ ]:
import os
import time
import pandas as pd
import paramiko

prefix = os.getenv("LAB_NODE_PREFIX")
if not prefix and os.getenv("ATTEMPT_ID"):
    prefix = f"lab-{os.environ['ATTEMPT_ID']}"

def host(node):
    return f"{prefix}-{node}" if prefix else node

nodes = {
    "r1": {"host": host("r1"), "address": "10.50.0.1/30", "peer": "10.50.0.2"},
    "s1": {"host": host("s1"), "address": "10.50.0.2/30", "peer": "10.50.0.1"},
}
nodes

## 2. Выполнение команд по SSH

Узлы используют учебные credentials `student` / `student`. Функция возвращает код завершения, stdout и stderr каждой команды.

In [ ]:
def run_ssh(target, commands):
    client = paramiko.SSHClient()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    client.connect(target, username="student", password="student", look_for_keys=False, allow_agent=False)
    rows = []
    try:
        for command in commands:
            _, stdout, stderr = client.exec_command(command)
            code = stdout.channel.recv_exit_status()
            rows.append({
                "command": command,
                "code": code,
                "stdout": stdout.read().decode().strip(),
                "stderr": stderr.read().decode().strip(),
            })
    finally:
        client.close()
    return rows

### Задание 1

Назначьте каждому `eth1` адрес из словаря `nodes` и включите интерфейс. Затем выведите краткое состояние адресов.

In [ ]:
configuration_results = []
for name, node in nodes.items():
    commands = [
        f"sudo -n ip address replace {node['address']} dev eth1",
        "sudo -n ip link set eth1 up",
        "ip -brief address show eth1",
    ]
    for result in run_ssh(node["host"], commands):
        configuration_results.append({"node": name, **result})

pd.DataFrame(configuration_results)

### Задание 2

Проверьте ICMP-связность в обе стороны. В таблице должны быть `code = 0` и ответы без потерь.

In [ ]:
ping_results = []
for name, node in nodes.items():
    result = run_ssh(node["host"], [f"ping -c 3 -W 1 {node['peer']}"])[0]
    ping_results.append({"node": name, **result})

pd.DataFrame(ping_results)

## 3. Мониторинг по SNMP

Получите стандартный объект `sysName.0` (`1.3.6.1.2.1.1.5.0`) с обоих узлов.

In [ ]:
from pysnmp.entity.rfc3413.oneliner import cmdgen

cmd_gen = cmdgen.CommandGenerator()
system_name_oid = "1.3.6.1.2.1.1.5.0"
snmp_results = []

for name, node in nodes.items():
    error, status, index, binds = cmd_gen.getCmd(
        cmdgen.CommunityData("public"),
        cmdgen.UdpTransportTarget((node["host"], 161), timeout=2, retries=1),
        system_name_oid,
    )
    snmp_results.append({
        "node": name,
        "error": str(error or status or ""),
        "sysName": str(binds[0][1]) if binds and not error and not status else None,
    })

pd.DataFrame(snmp_results)

## 4. Проверка результата

Откройте терминал Codespace и выполните:

```bash
./scripts/lab check
```

Ожидаемый результат — `5/5 checks passed`. Checker вернёт тот же JSON-контракт, который production Clabgate передаёт в CMS и Moodle/LTI.